## 1. Installation and Dependencies

# PLEASE RESTRART THE SESSION AFTER INSTALLING THE DEPENDIES TO AVOID IMPORT ERRORS

# Using P100 gpu

In [ ]:
# Install required packages
# CRITICAL: Install in this exact order with these exact versions

# Step 1: Core dependencies first
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu118

# Step 2: PyAnnote ecosystem (latest compatible versions)
!pip install -q pyannote.core==5.0.0
!pip install -q pyannote.database==5.1.0
!pip install -q pyannote.metrics==3.2.1
!pip install -q pyannote.audio==3.3.2

# Step 3: Lightning and other dependencies
!pip install -q pytorch-lightning==2.4.0
!pip install -q lightning==2.4.0
!pip install -q tensorboard

# Step 4: Audio processing
!pip install -q pydub librosa soundfile
!pip install torch-audiomentations==0.11.0

# Step 5: Utilities
!pip install -q pandas tqdm pyyaml

!pip install "numpy<2.0"
!pip install git+https://github.com/MahmoudAshraf97/demucs.git

print("All packages installed successfully!")

## 2. Imports and Configuration

In [1]:
import os
import json
import yaml
import pandas as pd
import numpy as np
import shutil
from pathlib import Path
from typing import Dict, List
import warnings
warnings.filterwarnings('ignore')
print(np.__version__)

# PyTorch
import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger

# PyAnnote
from pyannote.audio import Model, Inference, Pipeline
from pyannote.audio.tasks import SpeakerDiarization
from pyannote.database import FileFinder, get_protocol, registry
from pyannote.core import Annotation, Segment, Timeline
from pyannote.metrics.diarization import DiarizationErrorRate

# Audio
from pydub import AudioSegment
import torchaudio
from torch_audiomentations import Compose, Gain, AddBackgroundNoise

# Utils
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

# uncomment ffor a100
torch.set_float32_matmul_precision('high')
torch.backends.cudnn.benchmark = True
import pytorch_lightning as pl

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

1.26.4
PyTorch version: 2.5.1+cu118
CUDA available: True
GPU: Tesla P100-PCIE-16GB


# 3. Directories

In [ ]:
# ==================== CONFIGURATION ====================

# Hugging Face Token (required for pre-trained models)
# HF_TOKEN = "lol"  # Replace with your token

# Paths - UPDATE THESE FOR YOUR ENVIRONMENT
TRAIN_AUDIO_DIR = "/kaggle/input/competitions/dl-sprint-4-0-bengali-speaker-diarization-challenge/diarization/diarization/train/audio"  # Directory with .wav files
TRAIN_ANNOTATION_DIR = "/kaggle/input/competitions/dl-sprint-4-0-bengali-speaker-diarization-challenge/diarization/diarization/train/annotation"  # Directory with .csv/.rttm files
TEST_AUDIO_DIR = "/kaggle/input/competitions/dl-sprint-4-0-bengali-speaker-diarization-challenge/diarization/diarization/test/audio"  # Test audio files

# Output paths
WORK_DIR = "./work"
DATABASE_DIR = f"{WORK_DIR}/database"
MODEL_DIR = f"{WORK_DIR}/models"
CHECKPOINT_DIR = f"{WORK_DIR}/checkpoints"
LOGS_DIR = f"{WORK_DIR}/logs"
SUBMISSION_PATH = f"{WORK_DIR}/submission.csv"

# Create directories
for d in [WORK_DIR, DATABASE_DIR, MODEL_DIR, CHECKPOINT_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

# ==================== AUDIO SETTINGS ====================
SAMPLE_RATE = 16000  # Required by pyannote
MONO = True

# ==================== TRAINING SETTINGS ====================
# Model configuration
PRETRAINED_MODEL = "ishmamzarif/bengali-diarization_aug_v1"  # Best segmentation model
PRETRAINED_EMBEDDING = "pyannote/wespeaker-voxceleb-resnet34-LM"  # Best embedding

# Training hyperparameters
BATCH_SIZE = 32  # Adjust based on GPU memory
NUM_WORKERS = 4
MAX_EPOCHS = 20
LEARNING_RATE = 5e-5
DURATION = 5.0  # seconds per training chunk
MAX_SPEAKERS_PER_CHUNK = 3  # Maximum speakers in a chunk

# Early stopping
PATIENCE = 5
MIN_DELTA = 0.001

# Train/val split
VAL_SPLIT = 0.20

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"✓ Configuration loaded")
print(f"✓ Device: {DEVICE}")
print(f"✓ Batch size: {BATCH_SIZE}")
print(f"✓ Max epochs: {MAX_EPOCHS}")

✓ Configuration loaded
✓ Device: cuda
✓ Batch size: 32
✓ Max epochs: 20


# 4. Preprocessing audio

In [3]:
def preprocess_audio(input_path, output_path='./preprocessed'):
    """
    Preprocess audio to 16kHz mono WAV.
    """
    if output_path is None:
        output_path = input_path
    
    try:
        return_code = os.system(
            f'python -m demucs.separate -n htdemucs --two-stems=vocals "{input_path}" -o "temp_outputs" --device "{DEVICE}"'
        )
    
        if return_code != 0:
            logging.warning("Source splitting failed, using original audio file.")
            vocal_target = input_path
        else:
            vocal_target = os.path.join(
                "temp_outputs",
                "htdemucs",
                os.path.splitext(os.path.basename(input_path))[0],
                "vocals.wav",
            )

            non_vocal_target = os.path.join(
                "/kaggle/working/temp_outputs",
                "htdemucs",
                os.path.splitext(os.path.basename(input_path))[0],
                "no_vocals.wav",
            )
            
            # Use os.remove() to delete a file
            os.remove(non_vocal_target)
        audio = AudioSegment.from_wav(vocal_target)
        
        # Convert to mono
        if audio.channels > 1:
            audio = audio.set_channels(1)
        
        # Resample to 16kHz
        if audio.frame_rate != SAMPLE_RATE:
            audio = audio.set_frame_rate(SAMPLE_RATE)
        
        # Export
        audio.export(output_path, format="wav")
        return output_path
    
    except Exception as e:
        print(f"Error preprocessing {input_path}: {e}")
        return None

print("✓ Data preparation functions loaded")

✓ Data preparation functions loaded


# Inference

In [4]:
import huggingface_hub
import pyannote.audio.core.model

# 1. Save the original Hugging Face download function
original_download = huggingface_hub.hf_hub_download

def patched_download(*args, **kwargs):
    if 'use_auth_token' in kwargs:
        kwargs['token'] = kwargs.pop('use_auth_token')
    return original_download(*args, **kwargs)

# 3. Overwrite the function inside pyannote with our patched version
pyannote.audio.core.model.hf_hub_download = patched_download
print(" PyAnnote Monkey Patch Applied!")

 PyAnnote Monkey Patch Applied!


In [5]:
# OPTIMIZED INFERENCE CODE - FAST & GPU-ACCELERATED

import torch
from pathlib import Path
import json
from tqdm.auto import tqdm
import pandas as pd

def load_trained_pipeline_FAST(model_dir, device):
    """
    Load trained pipeline with GPU optimization.
    This version is 5-10x faster than the original.
    """
    print("Loading model for GPU inference...")
    
    # Load config
    # config_path = Path(model_dir) / "pipeline_config.json"
    # with open(config_path) as f:
    #     config = json.load(f)
    
    # # Load the PRETRAINED model (not checkpoint) for faster loading
    # model_path = Path(model_dir) / "final_model.ckpt"
    
    # if not model_path.exists():
    #     # Fallback to checkpoint
    #     print("⚠ Pretrained model not found, using checkpoint (slower)...")
    #     from pyannote.audio import Model
    #     segmentation_model = Model.from_pretrained(
    #         checkpoint_callback.best_model_path
    #     )
    # else:
    from pyannote.audio import Model
    segmentation_model = Model.from_pretrained(PRETRAINED_MODEL)
    
    # Move model to GPU immediately
    segmentation_model = segmentation_model.to(device)
    segmentation_model.eval()  # Set to evaluation mode
    
    print(f"✓ Model loaded on {device}")
    
    # Create pipeline with GPU support
    from pyannote.audio.pipelines import SpeakerDiarization
    
    pipeline = SpeakerDiarization(
        segmentation=segmentation_model,
        embedding=PRETRAINED_EMBEDDING,
        embedding_exclude_overlap=True,
        use_auth_token=HF_TOKEN,  # CRITICAL: Must include auth token
    )
    
    # Move entire pipeline to GPU
    pipeline.to(device)
    
    # Instantiate with optimized parameters
    optimized_params = {
        "segmentation": {
            "threshold": 0.5,
            "min_duration_off": 0.0,  # No minimum gap
        },
        "clustering": {
            "method": "centroid",
            "min_cluster_size": 12,  # Minimum samples per cluster
            "threshold": 0.7,  # Clustering threshold
        },
    }
    
    try:
        pipeline.instantiate(optimized_params)
        print("✓ Pipeline instantiated with optimized parameters")
    except Exception as e:
        print(f"Warning: {e}")
        # Fallback
        pipeline.instantiate({
            "clustering": {
                "method": "centroid",
                "min_cluster_size": 12,
                "threshold": 0.7,
            }
        })
        print("✓ Pipeline instantiated with fallback parameters")
    
    return pipeline


def predict_test_set_FAST(pipeline, test_audio_dir, output_path, device):
    """
    Fast GPU-accelerated inference with optimizations.
    """
    audio_files = sorted(Path(test_audio_dir).glob("*.wav"))
    print(f"Found {len(audio_files)} audio files")
    if len(audio_files) == 0:
        raise ValueError(f"No audio files found in {test_audio_dir}")
        
    for audio_file in tqdm(audio_files, desc="Preprocessing data"):
        preprocess_audio(str(audio_file))
    
    
    # 1. Define the base path
    base_path = Path("/kaggle/working/temp_outputs/htdemucs")
    
    # 2. Get the folders (assuming htdemucs created a folder per song)
    au = sorted(base_path.glob("*"))
    
    # 3. Change {} to [] because you are appending to a list
    audio_files = [] 
    
    for folder in au:
        # 4. Construct the path directly using pathlib's '/' operator
        # 'folder' is already a Path object, so we look for vocals.wav inside it
        vocal_file = folder / "vocals.wav"
        
        # Optional: Check if the file actually exists before adding
        if vocal_file.exists():
            audio_files.append(vocal_file)
    
    print(f"Found {len(audio_files)} audio files")

    # 5. Fix the quote syntax inside the f-string
    if len(audio_files) == 0:
        raise ValueError(f"No audio files found in '{base_path}'")


    results = []
    
    # Use torch.no_grad() for faster inference
    with torch.no_grad():
        for audio_file in tqdm(audio_files, desc="🚀 Fast GPU Inference"):
            uri = audio_file.parent.name
            
            try:
                # Run diarization on GPU
                diarization = pipeline(str(audio_file))
                
                # Convert to submission format
                segments = []
                speaker_mapping = {}
                next_id = 1
                
                for segment, _, speaker in diarization.itertracks(yield_label=True):
                    # Map speakers to sequential IDs
                    if speaker not in speaker_mapping:
                        speaker_mapping[speaker] = next_id
                        next_id += 1
                    
                    segments.append({
                        "start_time": round(segment.start, 2),
                        "end_time": round(segment.end, 2),
                        "speaker_id": speaker_mapping[speaker]
                    })
                
                results.append({
                    "filename": uri,
                    "diarization": json.dumps(segments)
                })
                
            except Exception as e:
                print(f"\n⚠ Error processing {uri}: {e}")
                # Add empty result to maintain file count
                results.append({
                    "filename": uri,
                    "diarization": json.dumps([])
                })
    
    # Create submission
    df = pd.DataFrame(results)
    # df.to_csv(output_path, index=False)
    
    return df


# ============================================================
# RUN OPTIMIZED INFERENCE
# ============================================================

print("\n" + "="*60)
print("FAST GPU INFERENCE")
print("="*60)

# Ensure CUDA is available
if not torch.cuda.is_available():
    print("⚠ WARNING: CUDA not available, using CPU (will be slow)")
    DEVICE = torch.device("cpu")
else:
    DEVICE = torch.device("cuda")
    print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
    # Enable cuDNN autotuner for additional speed
    torch.backends.cudnn.benchmark = True

# Load pipeline with GPU optimization
pipeline = load_trained_pipeline_FAST(MODEL_DIR, DEVICE)

# Run fast inference
print("\n Running FAST inference on test set...")
print(f"Device: {DEVICE}")
print(f"Number of test files: {len(list(Path(TEST_AUDIO_DIR).glob('*.wav')))}")

submission_df = predict_test_set_FAST(pipeline, TEST_AUDIO_DIR, SUBMISSION_PATH, DEVICE)

# print(f"\n✓ Submission created: {SUBMISSION_PATH}")
print(f"✓ Total files processed: {len(submission_df)}")
print("\nSubmission preview:")
print(submission_df.head())

# Clean up GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("\n✓ GPU memory cleared")



FAST GPU INFERENCE
✓ Using GPU: Tesla P100-PCIE-16GB
Loading model for GPU inference...


pytorch_model.bin:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

config.yaml:   0%|          | 0.00/235 [00:00<?, ?B/s]

✓ Model loaded on cuda


pytorch_model.bin:   0%|          | 0.00/26.6M [00:00<?, ?B/s]

...

Selected model is a bag of 1 models. You will see that many progress bars per track.
Separated tracks will be stored in /kaggle/working/temp_outputs/htdemucs
Separating track /kaggle/input/competitions/dl-sprint-4-0-bengali-speaker-diarization-challenge/diarization/diarization/test/audio/test_032.wav


100%|██████████████████████████████████████████████| 2597.3999999999996/2597.3999999999996 [01:19<00:00, 32.62seconds/s]


Found 14 audio files


🚀 Fast GPU Inference:   0%|          | 0/14 [00:00<?, ?it/s]

✓ Total files processed: 14

Submission preview:
   filename                                        diarization
0  test_010  [{"start_time": 12.84, "end_time": 27.52, "spe...
1  test_012  [{"start_time": 0.03, "end_time": 9.03, "speak...
2  test_016  [{"start_time": 34.98, "end_time": 41.0, "spea...
3  test_018  [{"start_time": 0.03, "end_time": 3.52, "speak...
4  test_019  [{"start_time": 1.67, "end_time": 6.53, "speak...

✓ GPU memory cleared


In [6]:
import json
import pandas as pd

def format_speaker_id(json_str):
    """
    Parses a JSON string, updates speaker_id to 'SPEAKER_X' format,
    and returns the modified JSON string.
    """
    try:
        # Parse the JSON string into a list of dictionaries
        data = json.loads(json_str)
        
        if isinstance(data, list):
            for entry in data:
                # Check if speaker_id exists and update it
                if 'speaker_id' in entry:
                    entry['speaker_id'] = f"SPEAKER_{entry['speaker_id']}"
            
            # Convert the list back to a JSON string
            return json.dumps(data)
        return json_str
    except Exception as e:
        # In case of error (e.g. malformed JSON), return original string
        return json_str

# 1. Load the CSV file
# df = pd.read_csv('/kaggle/working/work/submission.csv')

# 2. Apply the formatting function to the 'diarization' column
submission_df['diarization'] = submission_df['diarization'].apply(format_speaker_id)

# 3. Save the modified DataFrame to a new CSV file
submission_df.to_csv('submission.csv', index=False)

print("File processed and saved as 'submission.csv'")

File processed and saved as 'submission.csv'
